Import necessary libraries for training. Ultralytics for model weights, WandB for tracking. If wandB isnt downloaded run: pip install -U ultralytics wandb

In [ ]:
import torch
import os
os.environ["WANDB_MODE"] = "online"
import wandb
import yaml
from pathlib import Path
import json
import pandas as pd
import ray
from ray import tune
from ray.tune.search.optuna import OptunaSearch
import gc
from ultralytics import YOLO, settings
from ultralytics.models.yolo.detect import DetectionTrainer
settings.update({"wandb": True})
print("WandB enabled:", settings["wandb"])
print(torch.cuda.is_available())  # should print True
print(torch.cuda.get_device_name(0))
 # should print your GPU name

Hyperparameter tuning using Optuna from RayTune

In [ ]:
#Hyperparameter tuning
# Limit Ray object store to prevent RAM exhaustion
if ray.is_initialized():
    ray.shutdown()

    # Clear CUDA cache and inter-process handles
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()  # wait for any pending CUDA ops to actually finish

print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

#ray can be resource intensive, so we limit the object store memory to 2GB and specify a temporary directory for Ray to use. Adjust the path as needed.

ray.init(object_store_memory=2_000_000_000, _temp_dir="")  # replace with your temp directory path

wandb.init(project="", #replace with your project name
           id=None,
           resume="never",
           name="") #replace with your run name

model = YOLO() #path to the pretrained YOLO26x model, e.g., "yolo26x.pt"

#set dataset path
dataset = #path to the dataset.yaml file

searcher= OptunaSearch(metric="metrics/mAP50(B)", mode="max")

result_grid = model.tune(data=dataset, epochs=50, batch=8, iterations=15, max_det=800,
                          search_alg=searcher, optimizer="MuSGD", #set optimizer to MuSGD to mimic real training conditions
                          project="",  #replace with your project name
                         name="",  #replace with your run name
                         use_ray=True, gpu_per_trial=1)

best = result_grid.get_best_result(metric="metrics/mAP50(B)", mode="max")
print(best.config)

# --- Force cleanup ---

# 1. Finish W&B run explicitly
wandb.finish()

# 2. Shut down Ray — this kills worker actor processes and frees their GPU contexts
if ray.is_initialized():
    ray.shutdown()

# 3. Clear PyTorch's CUDA cache and run garbage collection
del model, result_grid, best  # drop references so gc can actually collect them
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()  # cleans up any leftover inter-process CUDA memory handles

print("Cleanup complete.")

Save optimal hyperparameters to yaml

In [ ]:
TUNING_DIR = Path("") #replace with the path to your tuning results directory
OUTPUT_YAML = Path("") #replace with the path where you want to save the best hyperparameters YAML file

In [ ]:
MAP_COL = "metrics/mAP50(B)"

results = []

for trial_dir in sorted(TUNING_DIR.glob("_tune_*")):
    progress_csv = trial_dir / "progress.csv"
    params_json  = trial_dir / "params.json"

    if not progress_csv.exists() or not params_json.exists():
        continue

    df = pd.read_csv(progress_csv)

    if MAP_COL not in df.columns:
        print(f"[SKIP] {trial_dir.name} — column '{MAP_COL}' not found")
        continue

    # Drop rows where mAP50 is 0 (failed / warmup epochs with no detections)
    valid = df[df[MAP_COL] > 0]
    if valid.empty:
        print(f"[SKIP] {trial_dir.name} — no valid mAP50 values")
        continue

    best_row   = valid.loc[valid[MAP_COL].idxmax()]
    best_map50 = best_row[MAP_COL] #find which epoch in which iteration had the best mAP50 value
    best_epoch = int(best_row["epoch"])

    results.append({
        "trial_id":   trial_dir.name,
        "best_mAP50": best_map50,
        "best_epoch": best_epoch,
        "params_path": params_json,
    })

    print(f"{trial_dir.name}  |  best mAP50 = {best_map50:.5f}  (epoch {best_epoch})")

print(f"\nTotal trials scanned: {len(results)}")

In [ ]:
if not results:
    raise RuntimeError("No valid trial results found — check the tuning directory path.")

best = max(results, key=lambda r: r["best_mAP50"])

print(f"Best trial : {best['trial_id']}")
print(f"Best mAP50 : {best['best_mAP50']:.5f}  (epoch {best['best_epoch']})")

In [ ]:
with open(best["params_path"]) as f:
    params = json.load(f)

# Attach metadata so the YAML is self-documenting
output = {
    "_meta": {
        "trial_id":   best["trial_id"],
        "best_mAP50": round(best["best_mAP50"], 6),
        "best_epoch": best["best_epoch"],
        "source_dir": str(TUNING_DIR),
    },
    **params,
}

with open(OUTPUT_YAML, "w") as f:
    yaml.dump(output, f, default_flow_style=False, sort_keys=False)

print(f"Saved to: {OUTPUT_YAML}")
print()
print(yaml.dump(output, default_flow_style=False, sort_keys=False))

In [ ]:
#set device to gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#set dataset path
dataset = #path to the dataset.yaml file

class CustomSaveTrainer(DetectionTrainer):
    """Trainer that saves the best model based on mAP@0.5 instead of default fitness."""

    def validate(self):
        """Override fitness to use mAP@0.5 for best model selection."""
        metrics, fitness = super().validate()
        if metrics:
            fitness = metrics.get("metrics/mAP50(B)", fitness)
            if self.best_fitness is None or fitness > self.best_fitness:
                self.best_fitness = fitness
        return metrics, fitness

# Load best hyperparameters
#Remove metadata from yaml before loading here
with open("# path to best hyperparameters file", "r") as f: #replace with the path to your best hyperparameters file
    best_params = yaml.safe_load(f)

model = YOLO() #path to the pretrained YOLO26x model, e.g., "yolo26x.pt"

results = model.train(
    data=dataset,
    epochs=200,
    optimizer="MuSGD",
    batch=8,
    max_det=800,
    project="",  #replace with your project name
    name="",  #replace with your run name
    patience=50,
    trainer=CustomSaveTrainer,
    **best_params  # unpack all remaining hyperparameters
)

# Evaluate the model's performance on the validation set
model.val(max_det=800) 

# Finish the W&B run
wandb.finish()

In [ ]:
#evaluate the best model on the test set
dataset = #path to the dataset.yaml file
best_model = YOLO()  #path to the best model weights, e.g., "best.pt"
test_results = best_model.val(data=dataset, split='test', max_det=800)